<a href="https://colab.research.google.com/github/andresmqz/crypto-news-classifier/blob/main/crypto_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crypto headline dataset

**Daily routine:** `Runtime → Run all`, then label in cell 3. Type `q` to stop.
Everything is saved to Google Drive (`MyDrive/crypto-classifier/`), so stopping or disconnecting loses nothing.

## Labeling guide (v2)

### General
* Label the **main event** in the headline (who did what), not the background.
* Label from the **headline alone**. Knowing who named entities are is fine; don't add events or facts the headline doesn't state.
* Ask **"what changed in the world?"** If nothing (opinion, commentary, someone noticing something) → skip.
* **Two events in one headline:** label the one that leads. If unclear, a completed event beats a planned one.
* **Label the topic, not whether it happened:** planned, rumored, suspected or denied events get the same label as the event itself.
* Text after a colon naming a person or outlet (`: Bitwise`, `: Report`, `: 9 outlets compared`) is the source → ignore it.
* "Regulator says..." reporting on market activity → label the activity. Regulator acting (approves, rejects, proposes, investigates) → regulation.

### Skip
* Opinion pieces, explainers, listicles, analyst forecasts, headlines with no single event.
* Roundups, newsletters and podcast/video episode titles (`Morning Minute`, `what happened in crypto today`, titles with several `!`).
* Promotional content (presale reviews, "best tokens to buy", sponsored launch pages). Tip: no verb saying what happened = probably not news.
* Non-crypto news (general tech, AI, macro, cybersecurity) unless it directly involves crypto.
* Individuals (KOLs, traders, influencers) joining or leaving organizations; executive hires, firings or departures within one company.
* Crypto political campaign spending (PACs, election ads, donations). *(confirm)*

### Labels
1. **hack, exploit or security breach**: crypto protocols, exchanges, wallets or users losing funds or being breached, incl. suspected/denied hacks. Cryptojacking (malware mining crypto on victims' machines) = hack. *(confirm)*
2. **government regulation, taxation, or legal action**: laws proposed, approved or coming into force; tax changes; decisions on a legal basis (arrests, lawsuits, regulator approvals); government investigations and probes; bankruptcy/repayment proceedings (Mt. Gox, FTX); lobbying, comments or pushback on proposed rules.
3. **partnership or collaboration**: two or more organizations working together, not expansions. An executive joining another organization's body = partnership between their organizations. Crypto firm + traditional institution: if the deal between them is the news = partnership.
4. **acquisition or merger**: two companies becoming one, or one taking control of another (incl. explored/denied deals).
5. **new token or coin launch**: launch of a new crypto asset or an event about it (TGE, sale, airdrop, listing date).
6. **new product or feature launch**: a company releasing a product, tool, service or feature; an existing product expanding to a new chain, market or client group; network upgrades, hard forks, testnet/mainnet launches (incl. delays); governance votes on protocol changes; a traditional institution launching a specific crypto product.
7. **price movement or market trading**: explicit price moves, markets up or down, aggregate flows (ETF flows); on-chain and market metrics (TVL, staking, yields, network activity, hashrate, mining difficulty, stablecoin supply).
8. **institutional adoption or major industry shift**: companies or governments considering, buying, holding or using crypto; a named institution buying = adoption (vs aggregate flows = price); companies raising money to buy crypto for their treasury; governments buying/using crypto (making rules = regulation); institutions restricting, rejecting or stepping back from crypto.
9. **funding or investment**: money invested into a crypto company or protocol (VC rounds, strategic investments). If the investor takes control → acquisition.

## 1 · Connect Drive and set file paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DIR = "/content/drive/MyDrive/crypto-classifier"
os.makedirs(DIR, exist_ok=True)
POOL = f"{DIR}/pool.csv"        # everything collected
LABELED = f"{DIR}/labeled.csv"  # your labels

Mounted at /content/drive


## 2 · Collect headlines
Regular feeds (normal news mix, `via = rss`) plus targeted searches for rare labels (`via = search: ...`).

In [2]:
!pip install -q feedparser
import feedparser, pandas as pd, datetime
from urllib.parse import quote_plus

def google_news_feed(query):
    return f"https://news.google.com/rss/search?q={quote_plus(query)}&hl=en-US&gl=US&ceid=US:en"

FEEDS = {
    "coindesk": "https://www.coindesk.com/arc/outboundfeeds/rss/",
    "cointelegraph": "https://cointelegraph.com/rss",
    "decrypt": "https://decrypt.co/feed",
    "theblock": "https://www.theblock.co/rss.xml",
}

SEARCHES = [
    "TGE token generation event when:7d",                    # token launch
    "crypto token sale when:7d",                             # token launch
    "crypto exploit hack when:7d",                           # hack
    '"crypto" raises million seed OR "Series A" when:7d',    # funding
    'crypto firm "to acquire" when:7d',                      # acquisition
    "crypto exchange acquisition deal when:7d",              # acquisition
    "crypto partnership when:7d",                            # partnership
]
PER_SEARCH = 15
today = datetime.date.today().isoformat()

rows = []

for source, url in FEEDS.items():
    feed = feedparser.parse(url)
    for entry in feed.entries:
        rows.append({"headline": entry.title.strip(), "source": source,
                     "published": entry.get("published", ""),
                     "collected": today, "via": "rss"})

for q in SEARCHES:
    feed = feedparser.parse(google_news_feed(q))
    for entry in feed.entries[:PER_SEARCH]:
        title, _, outlet = entry.title.rpartition(" - ")
        rows.append({"headline": (title or entry.title).strip(),
                     "source": outlet or "google news",
                     "published": entry.get("published", ""),
                     "collected": today, "via": f"search: {q}"})

new = pd.DataFrame(rows)
new = new[new.headline.str.count("!") < 2]   # drop podcast/video episode titles

pool = pd.concat([pd.read_csv(POOL), new]) if os.path.exists(POOL) else new
pool["via"] = pool["via"].fillna("rss")
pool = pool.drop_duplicates("headline")
pool.to_csv(POOL, index=False)

print(f"Pool size: {len(pool)}\n")
print("Collected today:")
print(pool[pool.collected == today].via.value_counts())

Pool size: 694

Collected today:
via
rss                                                           90
search: crypto partnership when:7d                             9
search: crypto firm "to acquire" when:7d                       8
search: TGE token generation event when:7d                     7
search: "crypto" raises million seed OR "Series A" when:7d     7
search: crypto exchange acquisition deal when:7d               5
search: crypto exploit hack when:7d                            3
search: crypto token sale when:7d                              2
Name: count, dtype: int64


## 3 · Label
Type a number and press Enter. `s` = skip, `q` = quit. Each label is saved to Drive immediately.

In [3]:
labels = [
    "hack, exploit or security breach",
    "government regulation, taxation, or legal action",
    "partnership or collaboration",
    "acquisition or merger",
    "new token or coin launch",
    "new product or feature launch",
    "price movement or market trading",
    "institutional adoption or major industry shift",
    "funding or investment",
]

pool = pd.read_csv(POOL)
if os.path.exists(LABELED):
    done = pd.read_csv(LABELED)
else:
    done = pd.DataFrame(columns=["headline", "label", "source", "published"])

todo = pool[~pool.headline.isin(done.headline)]
print(f"{len(done)} labeled, {len(todo)} to go\n")
for i, l in enumerate(labels, 1):
    print(f"{i}. {l}")
print("s = skip, q = quit")

for _, row in todo.iterrows():
    ans = input(f"\n{row.headline}\n> ").strip().lower()
    if ans == "q":

        break
    elif ans == "s":
        label = "skip"
    elif ans.isdigit() and 1 <= int(ans) <= len(labels):
        label = labels[int(ans) - 1]
    else:
        print("Didn't understand, leaving it for later")
        continue
    done.loc[len(done)] = [row.headline, label, row.source, row.published]
    done.to_csv(LABELED, index=False)

688 labeled, 6 to go

1. hack, exploit or security breach
2. government regulation, taxation, or legal action
3. partnership or collaboration
4. acquisition or merger
5. new token or coin launch
6. new product or feature launch
7. price movement or market trading
8. institutional adoption or major industry shift
9. funding or investment
s = skip, q = quit

Live updates: Bitcoin sheds early gains; Strategy's STRC moves to daily dividends
> s

Fed Rate Decision: What Markets and Bitcoin Traders Can Expect
> s

JPMorgan, Coca-Cola Among 7 More Companies Objecting to OKX-ICE Tokenized Stocks
> 8

Avail launches ShieldTX privacy tool targeting copy-trading bots on Hyperliquid
> 6

Onchain credit firm Birch Hill raises $4M seed led by Multicoin Capital: Decrypt
> 9

Binance Brings Back Stock Trading Through Ondo Finance Partnership
> 3


## 4 · Progress

In [4]:
done = pd.read_csv(LABELED)
pool = pd.read_csv(POOL)
usable = (done.label != "skip").sum()

print(f"{len(done)} labeled · {usable} usable · {len(pool) - len(done)} left to label")
print(f"Skip rate: {(done.label == 'skip').mean():.0%}\n")
print(done.label.value_counts())

694 labeled · 483 usable · 0 left to label
Skip rate: 30%

label
skip                                                211
government regulation, taxation, or legal action     75
price movement or market trading                     73
institutional adoption or major industry shift       70
new product or feature launch                        61
hack, exploit or security breach                     52
partnership or collaboration                         48
acquisition or merger                                45
funding or investment                                39
new token or coin launch                             20
Name: count, dtype: int64


## 5 · Fix a label (run only when needed)
Paste the start of the headline and the correct label.

In [7]:
def relabel(headline_start, new_label):
    done = pd.read_csv(LABELED)
    match = done.headline.str.startswith(headline_start)
    if match.sum() == 0:
        print("No headline starts with that text, nothing changed")
        return
    done.loc[match, "label"] = new_label
    done.to_csv(LABELED, index=False)
    print(f"Changed {match.sum()} row(s) to: {new_label}")

# Example (remove the # to run):
#relabel("Robinhood taps Ripple", "partnership or collaboration")

Changed 1 row(s) to: partnership or collaboration
Changed 1 row(s) to: partnership or collaboration
Changed 2 row(s) to: hack, exploit or security breach
Changed 1 row(s) to: price movement or market trading


**skimming skips to find labelling errors:**

In [ ]:
done = pd.read_csv(LABELED)
skips = done[done.label == "skip"].reset_index(drop=True)
for i, row in skips.iterrows():
    print(f"{i:>3}  [{row.source}]  {row.headline}")

  0  [coindesk]  Crypto Long & Short: Why real-world assets need fixed-rate loans onchain
  1  [coindesk]  Why centralized exchanges refuse to list legacy crypto tokens without explicit SEC immunity
  2  [coindesk]  Bitcoin risk management grows more popular as the 'just bet on price' era fades
  3  [coindesk]  A company that vouches for websites had secret keys so weak that 8 of them were cracked
  4  [cointelegraph]  Here’s what happened in crypto today
  5  [cointelegraph]  Move over memecoins: Crypto ‘meme games’ coming, bagholders and all
  6  [cointelegraph]  PHP Foundation admits ‘structural failure’ delayed critical security fix
  7  [cointelegraph]  Trump didn‘t mention ‘crypto’ once in White House cybercrime roundtable
  8  [cointelegraph]  Treasury Secretary Bessent likens GENIUS Act to Lincoln-era banking reform
  9  [cointelegraph]  5 alternatives to Ledger to store your Bitcoin
 10  [cointelegraph]  Tech giants rush to block 8 eMudhra certs that could enable fake sites
 1

## 6 · Finding dupes. First prompt only shows them:

In [5]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer, util

done = pd.read_csv(LABELED)
news = done[done.label != "skip"].reset_index(drop=True)

model = SentenceTransformer("all-MiniLM-L6-v2")
emb = model.encode(news.headline.tolist(), convert_to_tensor=True)
sim = util.cos_sim(emb, emb)

pairs = []
for i in range(len(news)):
    for j in range(i + 1, len(news)):
        pairs.append((sim[i][j].item(), i, j))
pairs.sort(reverse=True)

for score, i, j in pairs[:40]:
    warn = "" if news.label[i] == news.label[j] else "  ⚠️ different labels"
    print(f"{score:.2f}{warn}\n   {news.headline[i]}\n   {news.headline[j]}\n")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.98
   Standard Chartered signs definitive deal to acquire Zodia Custody
   Standard Chartered signs definitive agreements to acquire Zodia Custody

0.97
   Fireblocks denies report of $2.5 billion acquisition talks with Visa
   Fireblocks Denies Report of $2.5B Visa Acquisition Talks

0.95
   South Korea weighs fourth delay of crypto gains tax, eyes 2029 start: report
   South Korea weighs fourth delay on its crypto gains tax, pushing it to 2029

0.95
   Bitcoin ETFs bleed $635M as October slides back into the red
   Bitcoin ETFs Bleed $636 Million as October Turns Red Again

0.94
   Zcash co-founder proposes ending grants funding, citing $125 million treasury
   Zcash co-founder proposes ending grants subsidy, citing $125 million treasury

0.94
   Standard Chartered signs definitive deal to acquire Zodia Custody
   Standard Chartered Signs Definitive Deal To Acquire 100% Of Zodia Custody Limited: 16 outlets compared

0.94
   Visa and Mesh Partner to Make Crypto and Stablecoin Balanc

In [6]:
for low in [0.80, 0.75, 0.70, 0.65, 0.60]:
    band = [p for p in pairs if low <= p[0] < low + 0.05][:4]
    print(f"===== {low:.2f} – {low + 0.05:.2f} =====\n")
    for score, i, j in band:
        warn = "" if news.label[i] == news.label[j] else "  ⚠️ different labels"
        print(f"{score:.2f}{warn}\n   {news.headline[i]}\n   {news.headline[j]}\n")

===== 0.80 – 0.85 =====

0.85
   SpaceX and Tesla explore merger that would unite more than 30,000 bitcoin
   SpaceX and Tesla Are Exploring a Merger That Would Hold 30,000 BTC. What Could It Mean for Bitcoin?

0.85
   Visa, Mesh partner to enable crypto spending at 175M merchant locations
   Mesh and Visa Partner to Make Stablecoin and Crypto Balances Spendable Across 175 Million Merchants

0.85
   Strategy Pauses Bitcoin Buying, Spends $205M Buying Back STRC
   Strategy pauses bitcoin buys, repurchases $205 million of STRC preferred shares

0.85
   A16z is the latest VC giant to widen bets beyond crypto and into AI and emerging tech
   a16z Drops Crypto From Its Name, Widens Bets to AI and Frontier Tech

===== 0.75 – 0.80 =====

0.80
   Standard Chartered Signs Definitive Deal To Acquire 100% Of Zodia Custody Limited: 16 outlets compared
   Standard Chartered to acquire Zodia, expanding crypto custody business

0.80
   Visa To Acquire Fireblocks For $2.5 Billion: Reports
   Visa in R

In [7]:
THRESHOLD = 0.75

group = list(range(len(news)))
def find(x):
    while group[x] != x:
        x = group[x]
    return x

for score, i, j in pairs:
    if score < THRESHOLD:
        break
    group[find(i)] = find(j)

news["story"] = [find(i) for i in range(len(news))]
news["keep"] = ~news.story.duplicated()

print(f"{len(news)} usable headlines → {news.keep.sum()} distinct stories\n")
print(news[news.keep].label.value_counts())

print("\nBiggest groups:")
for s in news.story.value_counts().head(5).index:
    copies = news[news.story == s]
    print(f"  {len(copies)}× {copies.headline.iloc[0]}")

news[["headline", "label", "story", "keep"]].to_csv(f"{DIR}/stories.csv", index=False)

483 usable headlines → 367 distinct stories

label
price movement or market trading                    68
government regulation, taxation, or legal action    58
new product or feature launch                       50
institutional adoption or major industry shift      48
partnership or collaboration                        41
hack, exploit or security breach                    38
funding or investment                               28
acquisition or merger                               20
new token or coin launch                            16
Name: count, dtype: int64

Biggest groups:
  13× Crypto custody specialist Fireblocks denies it's in talks to be acquired by Visa
  7× Standard Chartered signs definitive deal to acquire Zodia Custody
  6× Bitget acquires Cryptonomics from GMO to expand into Thailand
  6× Hackers Exploit Plex2Shell Flaws to Install Crypto Miners and Linux Proxy Backdoor
  6× Wells Fargo and five others object to OKX-ICE tokenized stock listings: Report


## 7 · Original 21

In [8]:
test21 = pd.DataFrame([
    ("France's parliament approves vote to hit stablecoin trades with 31% capital gains tax", "government regulation, taxation, or legal action"),
    ("Equity weakness could damp demand for bitcoin after biggest ETF inflows in a month", "price movement or market trading"),
    ("Bitcoin drops to $81,400 as Asian chip stocks tumble, options traders seek downside protection", "price movement or market trading"),
    ("South Korean IPO slump adds to AI market jitters as Brent crude tops $111", "price movement or market trading"),
    ("Bitcoin ETFs just saw their biggest day in a month thanks entirely to BlackRock", "price movement or market trading"),
    ("Cardano founder Charles Hoskinson is building a new programming language for the network", "new product or feature launch"),
    ("Goldman, Wells Fargo, Morgan Stanley see tokenization as a growth play, not a threat", "institutional adoption or major industry shift"),
    ("Deutsche Bank targets Asia crypto custody expansion in 2027 as German launch nears", "new product or feature launch"),
    ("SEC tokenized stock framework faces issuer pushback as Broadcom, Verizon object to OKX-ICE listing", "government regulation, taxation, or legal action"),
    ("Latvia eyes crypto tax payments in 2027 as parliament overhauls crypto taxation", "government regulation, taxation, or legal action"),
    ("Polygon sees Coinme acquisition as entry to payments sector dominated by Visa, Stripe", "acquisition or merger"),
    ("JPMorgan expands 'JPM Coin' deposit token to Ethereum for institutional clients", "new product or feature launch"),
    ("Farcaster acquired by Balaji's Network School, with new 'social smart contract' for users", "acquisition or merger"),
    ("Helius CEO Mert Mumtaz joins NEAR Foundation Council", "partnership or collaboration"),
    ("StarkWare launches OffMarket to let Polymarket users trade more privately", "new product or feature launch"),
    ("Swiss regulator FINMA approves country's first spot bitcoin ETF for retail investors", "government regulation, taxation, or legal action"),
    ("Cysic suspends bridge after attacker mints tokens and moves $727,000 to Ethereum", "hack, exploit or security breach"),
    ("Magic Eden legacy approvals leave $5.7 million in NFTs exposed to exploit before rescue", "hack, exploit or security breach"),
    ("Bitget starts phased withdrawal resumption following $388 million exploit", "hack, exploit or security breach"),
    ("Securitize launches 1:1-backed tokenized stocks, starting with Apple, Nvidia, Strategy and more", "new product or feature launch"),
    ("Squid launches native token QUID, plans public sale to start on June 30", "new token or coin launch"),
], columns=["headline", "label"])
test21.to_csv(f"{DIR}/test21.csv", index=False)

done = pd.read_csv(LABELED)
emb_test = model.encode(test21.headline.tolist(), convert_to_tensor=True)
emb_all = model.encode(done.headline.tolist(), convert_to_tensor=True)
sim21 = util.cos_sim(emb_test, emb_all)

overlap = []
for t in range(len(test21)):
    for d in range(len(done)):
        score = sim21[t][d].item()
        if score >= 0.65:
            overlap.append((score, test21.headline[t], done.headline[d]))
overlap.sort(reverse=True)

for score, t, d in overlap:
    flag = "LEAK" if score >= 0.75 else "check"
    print(f"{score:.2f} [{flag}]\n   test:    {t}\n   dataset: {d}\n")

leaks = pd.DataFrame([o for o in overlap if o[0] >= 0.75], columns=["score", "test_headline", "headline"])
leaks.to_csv(f"{DIR}/test21_overlap.csv", index=False)
print(f"{leaks.headline.nunique()} dataset headlines overlap with the original 21")

0.86 [LEAK]
   test:    Bitcoin ETFs just saw their biggest day in a month thanks entirely to BlackRock
   dataset: Bitcoin ETFs Post Their Best Day in Nearly a Month as BlackRock Does All the Buying

0.84 [LEAK]
   test:    Polygon sees Coinme acquisition as entry to payments sector dominated by Visa, Stripe
   dataset: Polygon Labs completes Coinme acquisition to challenge Visa and Stripe in global payments

0.82 [LEAK]
   test:    Swiss regulator FINMA approves country's first spot bitcoin ETF for retail investors
   dataset: Swiss Regulator Approves First Retail Fund That Holds Bitcoin Directly

0.79 [LEAK]
   test:    Polygon sees Coinme acquisition as entry to payments sector dominated by Visa, Stripe
   dataset: Polygon Takes Aim At Visa And Stripe After Closing Its Coinme Purchase

0.73 [check]
   test:    JPMorgan expands 'JPM Coin' deposit token to Ethereum for institutional clients
   dataset: JPMorgan's JPM Coin Arrives on Ethereum, Putting Bank Dollars on a Public Blockcha

In [9]:
extra = pd.DataFrame([
    (0.73, "JPMorgan expands 'JPM Coin' deposit token to Ethereum for institutional clients",
     "JPMorgan's JPM Coin Arrives on Ethereum, Putting Bank Dollars on a Public Blockchain"),
    (0.71, "France's parliament approves vote to hit stablecoin trades with 31% capital gains tax",
     "French Lawmakers Back Stablecoin Swap Tax by Just One Vote"),
], columns=["score", "test_headline", "headline"])

leaks = pd.concat([leaks, extra])
leaks.to_csv(f"{DIR}/test21_overlap.csv", index=False)
print(f"{len(leaks)} dataset headlines set aside")

6 dataset headlines set aside


## 8 · First classifier

In [ ]:
from sklearn.linear_model import LogisticRegression

leaks = pd.read_csv(f"{DIR}/test21_overlap.csv")
train = news[news.keep & ~news.headline.isin(leaks.headline)].reset_index(drop=True)
print(f"Training on {len(train)} distinct stories\n")

X = model.encode(train.headline.tolist())
y = train.label

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X, y)

test21 = pd.read_csv(f"{DIR}/test21.csv")
probs = clf.predict_proba(model.encode(test21.headline.tolist()))
ranked = probs.argsort(axis=1)[:, ::-1]

top1 = top2 = 0
for k, row in test21.iterrows():
    first, second = clf.classes_[ranked[k, 0]], clf.classes_[ranked[k, 1]]
    top1 += first == row.label
    top2 += row.label in (first, second)
    mark = "✅" if first == row.label else ("🟡" if second == row.label else "❌")
    print(f"{mark} 1st: {first} | 2nd: {second} | you: {row.label} | {row.headline}")

print(f"\nYour classifier — Top-1: {top1}/21   Top-2: {top2}/21")
print("BART:  12/21 · Claude: 20/21")

Training on 252 distinct stories

✅ 1st: government regulation, taxation, or legal action | 2nd: price movement or market trading | you: government regulation, taxation, or legal action | France's parliament approves vote to hit stablecoin trades with 31% capital gains tax
✅ 1st: price movement or market trading | 2nd: institutional adoption or major industry shift | you: price movement or market trading | Equity weakness could damp demand for bitcoin after biggest ETF inflows in a month
✅ 1st: price movement or market trading | 2nd: institutional adoption or major industry shift | you: price movement or market trading | Bitcoin drops to $81,400 as Asian chip stocks tumble, options traders seek downside protection
✅ 1st: price movement or market trading | 2nd: new product or feature launch | you: price movement or market trading | South Korean IPO slump adds to AI market jitters as Brent crude tops $111
✅ 1st: price movement or market trading | 2nd: institutional adoption or major indu

## 9 · Build dataset

In [10]:
done = pd.read_csv(LABELED)
pool = pd.read_csv(POOL)
leaks = pd.read_csv(f"{DIR}/test21_overlap.csv")

# a date for every headline: publish date, or collection day if missing
dates = pool[["headline", "collected"]]
done = done.merge(dates, on="headline", how="left")
done["date"] = pd.to_datetime(done.published, utc=True, errors="coerce", format="mixed")
done["date"] = done.date.fillna(pd.to_datetime(done.collected, utc=True))

# news: one headline per story, dated by the story's earliest headline
news_d = news.merge(done[["headline", "date"]], on="headline", how="left")
news_d = news_d[~news_d.headline.isin(leaks.headline)]
news_d["date"] = news_d.groupby("story").date.transform("min")
news_one = news_d[~news_d.story.duplicated()][["headline", "label", "date"]]

# noise: skips renamed, capped at 70
noise = done[done.label == "skip"].sample(n=70, random_state=42)[["headline", "date"]]
noise["label"] = "noise"

dataset = pd.concat([news_one, noise]).sort_values("date").reset_index(drop=True)
dataset.to_csv(f"{DIR}/dataset.csv", index=False)

print(f"{len(dataset)} examples, {dataset.date.isna().sum()} without a date\n")
print(dataset.label.value_counts())
print(f"\nOldest: {dataset.date.min()}\nNewest: {dataset.date.max()}")

431 examples, 0 without a date

label
noise                                               70
price movement or market trading                    67
government regulation, taxation, or legal action    56
new product or feature launch                       49
institutional adoption or major industry shift      48
partnership or collaboration                        41
hack, exploit or security breach                    38
funding or investment                               28
acquisition or merger                               18
new token or coin launch                            16
Name: count, dtype: int64

Oldest: 2025-12-29 16:10:19+00:00
Newest: 2026-10-28 15:00:02+00:00


In [11]:
days = dataset.date.dt.date.value_counts().sort_index()
print("Before Oct 15:", days[days.index < pd.Timestamp("2026-10-15").date()].sum())
print()
print(days[days.index >= pd.Timestamp("2026-10-15").date()])

Before Oct 15: 10

date
2026-10-15     5
2026-10-16    11
2026-10-17     7
2026-10-18     3
2026-10-19    23
2026-10-20    30
2026-10-21    62
2026-10-22    66
2026-10-23    48
2026-10-24    18
2026-10-25     6
2026-10-26    49
2026-10-27    55
2026-10-28    38
Name: count, dtype: int64


In [12]:
CUTOFF = pd.Timestamp("2026-10-27", tz="UTC")

train = dataset[dataset.date < CUTOFF].reset_index(drop=True)
test = dataset[dataset.date >= CUTOFF].reset_index(drop=True)

train.to_csv(f"{DIR}/train.csv", index=False)
test.to_csv(f"{DIR}/test.csv", index=False)

print(f"Train: {len(train)}   Test: {len(test)}\n")
print(pd.DataFrame({"train": train.label.value_counts(),
                    "test": test.label.value_counts()}).fillna(0).astype(int))

Train: 338   Test: 93

                                                  train  test
label                                                        
acquisition or merger                                16     2
funding or investment                                20     8
government regulation, taxation, or legal action     46    10
hack, exploit or security breach                     31     7
institutional adoption or major industry shift       40     8
new product or feature launch                        32    17
new token or coin launch                             12     4
noise                                                56    14
partnership or collaboration                         31    10
price movement or market trading                     54    13


In [13]:
from sklearn.linear_model import LogisticRegression

train = pd.read_csv(f"{DIR}/train.csv")
test = pd.read_csv(f"{DIR}/test.csv")

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(model.encode(train.headline.tolist()), train.label)

probs = clf.predict_proba(model.encode(test.headline.tolist()))
ranked = probs.argsort(axis=1)[:, ::-1]
test["first"] = clf.classes_[ranked[:, 0]]
test["second"] = clf.classes_[ranked[:, 1]]

top1 = (test["first"] == test.label).mean()
top2 = ((test["first"] == test.label) | (test["second"] == test.label)).mean()
print(f"Classifier on new test ({len(test)} headlines) — Top-1: {top1:.0%}   Top-2: {top2:.0%}\n")

print("Per label (top-1):")
print(test.assign(ok=test["first"] == test.label).groupby("label").ok.agg(["sum", "count"]))

test.to_csv(f"{DIR}/test_preds_classifier.csv", index=False)

Classifier on new test (93 headlines) — Top-1: 55%   Top-2: 70%

Per label (top-1):
                                                  sum  count
label                                                       
acquisition or merger                               2      2
funding or investment                               5      8
government regulation, taxation, or legal action    4     10
hack, exploit or security breach                    7      7
institutional adoption or major industry shift      3      8
new product or feature launch                       4     17
new token or coin launch                            3      4
noise                                               4     14
partnership or collaboration                        8     10
price movement or market trading                   11     13


In [14]:
wrong = test[test["first"] != test.label]
print(pd.crosstab(wrong.label, wrong["first"]))

first                                             acquisition or merger  \
label                                                                     
funding or investment                                                 0   
government regulation, taxation, or legal action                      1   
institutional adoption or major industry shift                        0   
new product or feature launch                                         0   
new token or coin launch                                              0   
noise                                                                 0   
partnership or collaboration                                          0   
price movement or market trading                                      0   

first                                             funding or investment  \
label                                                                     
funding or investment                                                 0   
government regulation, t

In [15]:
!pip install -q anthropic
import anthropic, json
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
MODEL = "claude-haiku-5-5"

NOISE_TEXT = "not a crypto news event (opinion, analysis, explainer, promotion, roundup, or non-crypto news)"
names = sorted(train.label.unique())
shown = [NOISE_TEXT if l == "noise" else l for l in names]

system = "You classify crypto news headlines into exactly these categories:\n" + \
    "\n".join(f"- {l}" for l in shown) + \
    '\n\nReply with JSON only, no other text:\n{"first": "<best label>", "second": "<second-best label>"}\n' + \
    "Use the label text exactly as written above. first and second must be different."

def classify(headline):
    r = client.messages.create(model=MODEL, max_tokens=1024, system=system,
                               messages=[{"role": "user", "content": headline}])
    texts = [b.text for b in r.content if b.type == "text"]
    if not texts:
        return None, None
    text = texts[0].strip().removeprefix("```json").removesuffix("```").strip()
    try:
        out = json.loads(text)
        back = lambda x: "noise" if x == NOISE_TEXT else x
        return back(out.get("first")), back(out.get("second"))
    except json.JSONDecodeError:
        return None, None

claude = test[["headline", "label"]].copy()
claude[["first", "second"]] = [classify(h) for h in claude.headline]

top1 = (claude["first"] == claude.label).mean()
top2 = ((claude["first"] == claude.label) | (claude["second"] == claude.label)).mean()
invalid = (~claude["first"].isin(names)).sum()
print(f"Claude on new test — Top-1: {top1:.0%}   Top-2: {top2:.0%}   Invalid: {invalid}\n")
print(claude.assign(ok=claude["first"] == claude.label).groupby("label").ok.agg(["sum", "count"]))

claude.to_csv(f"{DIR}/test_preds_claude.csv", index=False)

Claude on new test — Top-1: 77%   Top-2: 86%   Invalid: 0

                                                  sum  count
label                                                       
acquisition or merger                               2      2
funding or investment                               8      8
government regulation, taxation, or legal action    8     10
hack, exploit or security breach                    7      7
institutional adoption or major industry shift      5      8
new product or feature launch                      11     17
new token or coin launch                            4      4
noise                                               7     14
partnership or collaboration                       10     10
price movement or market trading                   10     13


In [16]:
from transformers import pipeline

bart = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

rows = []
for h in test.headline:
    r = bart(h, candidate_labels=shown)
    first, second = [("noise" if l == NOISE_TEXT else l) for l in r["labels"][:2]]
    rows.append((first, second))

bart_preds = test[["headline", "label"]].copy()
bart_preds[["first", "second"]] = rows

top1 = (bart_preds["first"] == bart_preds.label).mean()
top2 = ((bart_preds["first"] == bart_preds.label) | (bart_preds["second"] == bart_preds.label)).mean()
print(f"BART on new test — Top-1: {top1:.0%}   Top-2: {top2:.0%}\n")
print(bart_preds.assign(ok=bart_preds["first"] == bart_preds.label).groupby("label").ok.agg(["sum", "count"]))

bart_preds.to_csv(f"{DIR}/test_preds_bart.csv", index=False)

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

BART on new test — Top-1: 45%   Top-2: 58%

                                                  sum  count
label                                                       
acquisition or merger                               1      2
funding or investment                               8      8
government regulation, taxation, or legal action    2     10
hack, exploit or security breach                    7      7
institutional adoption or major industry shift      3      8
new product or feature launch                       5     17
new token or coin launch                            1      4
noise                                               0     14
partnership or collaboration                        8     10
price movement or market trading                    7     13
